<a href="https://colab.research.google.com/github/telvinpaul1/montra_guest_experience_sentiment_project/blob/main/04_birtopic_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!git clone https://github.com/telvinpaul1/montra_guest_experience_sentiment_project.git
%cd /content/montra_guest_experience_sentiment_project

Cloning into 'montra_guest_experience_sentiment_project'...
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 9 (delta 0), reused 9 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (9/9), 16.70 KiB | 16.70 MiB/s, done.
/content/montra_guest_experience_sentiment_project


In [5]:
!pip install bertopic sentence-transformers umap-learn hdbscan

In [6]:
from google.colab import files

# Upload a file from your computer
uploaded = files.upload()

Saving montra_cleaned_data.csv to montra_cleaned_data.csv


In [7]:
# Import libraries
import re
import pandas as pd
import numpy as np

In [18]:
# Define the processed data directory
DATA_DIR = Path("../data/processed")

# Create the directory if it does not exist
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Processed data directory:", DATA_DIR.resolve())

Processed data directory: /content/data/processed


In [8]:
merged = pd.read_csv("montra_cleaned_data.csv")
merged.head()

,review_id,booking_id,guest_id,hotel_id,review_text,rating_score,review_date,sentiment_label,review_length,room_type,...,stay_duration,customer_type,loyalty_status,country,previous_stays,hotel_name,location,star_rating,total_rooms,cleaned_review_text
0,1,1,6699,18,The overall price was fair for what we paid. M...,3.2,2025-06-02,neutral,80,Suite,...,1,Leisure,NaN,France,4,The Aberdeen Grand,Aberdeen,4,118,the overall price was fair for what we paid mi...
1,2,2,11380,23,The room was nothing out of the ordinary for w...,4.0,2025-01-22,positive,160,Deluxe,...,4,Business,Gold,Germany,4,Nottingham Park Hotel,Nottingham,4,95,the room was nothing out of the ordinary for w...
2,3,3,4812,13,"Unfortunately, the pool was outdated — it clea...",1.9,2025-03-13,negative,102,Executive,...,2,Business,NaN,Ireland,4,The London Boutique,London,4,231,unfortunately the pool was outdated it clearly...
3,4,4,982,12,We were disappointed that the neighborhood was...,2.2,2026-03-18,negative,129,Family Room,...,3,Leisure,NaN,Nigeria,0,Manchester Metropolitan,Manchester,3,101,we were disappointed that the neighborhood was...
4,5,5,615,5,The room service was standard hotel fare for w...,3.3,2025-03-03,neutral,196,Deluxe,...,5,Group,NaN,United States,9,Glasgow Metropolitan,Glasgow,4,214,the room service was standard hotel fare for w...


In [9]:
# Convert cleaned review text into a list of documents
document = merged['cleaned_review_text'].astype(str).tolist()

# Print the total number of documents
print(f'Total documents: {len(document)}')

# Display the first three documents
document[:3]


Total documents: 35000


['the overall price was fair for what we paid might consider it again might not',
 'the room was nothing out of the ordinary for what we paid the overall price exceeded our expectations a great deal and the quality easily matched the price',
 'unfortunately the pool was outdated it clearly needed some upkeep wouldn t recommend it to others']

In [10]:
# List of UK cities to remove
UK_Cities = [
    'Birmingham' 'London' 'Oxford' 'Glasgow' 'Leeds' 'Belfast' 'Edinburgh'
    'Southampton' 'Nottingham' 'Manchester' 'Liverpool' 'Leicester'
    'Aberdeen' 'Brighton'
]

# Compile a pattern to match the city names (case-insensitive)
city_pattern = re.compile(
    r'\b(?:' + '|'.join(map(re.escape, UK_Cities)) + r')\b',
    re.IGNORECASE
)

# Remove city names from each document
document = [
    city_pattern.sub('', doc).strip()
    for doc in document
]

# Remove extra whitespace
document = [
    re.sub(r'\s+', ' ', doc).strip()
    for doc in document
]

# Display the first three cleaned documents
document[:3]

['the overall price was fair for what we paid might consider it again might not',
 'the room was nothing out of the ordinary for what we paid the overall price exceeded our expectations a great deal and the quality easily matched the price',
 'unfortunately the pool was outdated it clearly needed some upkeep wouldn t recommend it to others']

In [25]:
# Import libraries
from collections import Counter
import torch
from sentence_transformers import SentenceTransformer
from pathlib import Path
# Import BERTopic and clustering components
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
# Import CountVectorizer
from sklearn.feature_extraction.text import CountVectorizer


In [12]:
# Normalise documents
normalized_docs = [doc.lower().strip() for doc in document]

# Count exact duplicates
total_docs = len(normalized_docs)
unique_docs = len(set(normalized_docs))
duplicated_docs = total_docs - unique_docs

# Display results
print(f"Total documents: {total_docs}")
print(f"Unique documents: {unique_docs}")
print(f"Duplicated documents: {duplicated_docs}")

# Calculate duplicate document rate
if total_docs > 0:
    duplicated_rate = (duplicated_docs / total_docs) * 100
    print(f'Duplicated rate: {duplicated_rate:.2f}%')
else:
    print("No documents available to calculate the duplicate rate.")

Total documents: 35000
Unique documents: 34536
Duplicated documents: 464
Duplicated rate: 1.33%


In [14]:
# Calculate vocabulary size
all_words = " ".join(normalized_docs).split()

print(f"Vocab size: {len(set(all_words))}")

# Display the 20 most common words
print(Counter(all_words).most_common(20))

Vocab size: 521
[('the', 73552), ('was', 54683), ('we', 32665), ('nothing', 24495), ('our', 23913), ('in', 23668), ('to', 18777), ('for', 18311), ('what', 17937), ('a', 17697), ('and', 17560), ('that', 16124), ('about', 12575), ('overall', 12124), ('it', 11539), ('paid', 11056), ('stay', 11027), ('as', 11012), ('out', 10561), ('i', 10224)]


In [16]:
# Select device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Display selected device
print("Using device:", device)

Using device: cuda


In [19]:
# Load the embedding model
embedding_model = SentenceTransformer(
    "all-mpnet-base-v2",
    device=device
)

# Define the embedding cache path
EMB_PATH = DATA_DIR / "review-embeddings.npy"

# Load cached embeddings if available
if EMB_PATH.exists():
    embeddings = np.load(EMB_PATH)
    print("Loaded cached embeddings:", embeddings.shape)

else:
    # Generate embeddings from review documents
    embeddings = embedding_model.encode(
        document,
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    # Save embeddings for reuse
    np.save(EMB_PATH, embeddings)
    print("Computed and saved embeddings:", embeddings.shape)

# Check that embeddings match the document count
print("Documents:", len(document))
print("Embedding rows:", embeddings.shape[0])
print("Embedding dimensions:", embeddings.shape[1])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/547 [00:00<?, ?it/s]

Computed and saved embeddings: (35000, 768)
Documents: 35000
Embedding rows: 35000
Embedding dimensions: 768


In [21]:
# Display the first two review embeddings
embeddings[:2]

array([[-0.00413856,  0.01629657,  0.01087651, ...,  0.00215244,
         0.00641585, -0.01160033],
       [-0.00784516, -0.04081137,  0.01082292, ...,  0.01583503,
         0.04026258,  0.02184933]], dtype=float32)

In [23]:
# Configure UMAP for dimensionality reduction
umap_model = UMAP(
    n_neighbors=15,      # Number of neighbouring points considered
    n_components=5,      # Reduce embeddings to 5 dimensions
    min_dist=0.0,        # Allow closely grouped points
    metric="cosine",     # Measure cosine distance between embeddings
    random_state=42      # Ensure reproducible results
)

In [24]:
# Configure HDBSCAN for clustering
hdbscan_model = HDBSCAN(
    min_cluster_size=150,          # Minimum size of each cluster
    min_samples=15,               # Controls how strictly points are assigned
    metric='euclidean',            # Distance metric for clustering
    cluster_selection_method='eom', # Select stable clusters
    prediction_data=True           # Generate data for approximate predictions
)

In [28]:
# Configure CountVectorizer
vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 5),
    min_df=5
)
# Initialise the BERTopic model
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    calculate_probabilities=True,
    verbose=True
)

# Check the vectorizer before fitting
print(topic_model.vectorizer_model)

# Fit BERTopic using review documents and embeddings
topics, probabilities = topic_model.fit_transform(
    document,
    embeddings
)

# Check the vectorizer after fitting
print(topic_model.vectorizer_model)

2026-10-27 00:24:47,860 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


CountVectorizer(min_df=5, ngram_range=(1, 5), stop_words='english')


2026-10-27 00:25:47,941 - BERTopic - Dimensionality - Completed ✓
2026-10-27 00:25:47,943 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-10-27 00:26:03,586 - BERTopic - Cluster - Completed ✓
2026-10-27 00:26:03,601 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-10-27 00:26:05,917 - BERTopic - Representation - Completed ✓


CountVectorizer(min_df=5, ngram_range=(1, 5), stop_words='english')


In [29]:
# Reduce the number of topics automatically
topic_model.reduce_topics(
    document,
    nr_topics="auto"
)

2026-10-27 00:32:53,362 - BERTopic - Topic reduction - Reducing number of topics
2026-10-27 00:32:53,403 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-10-27 00:32:55,716 - BERTopic - Representation - Completed ✓
2026-10-27 00:32:55,722 - BERTopic - Topic reduction - Reduced number of topics from 70 to 21


In [30]:
# Display the topic distribution
topic_info = topic_model.get_topic_info()
display(topic_info)

# Count reviews assigned to each topic
print("Total reviews:", len(topics))
print("Number of topics excluding outliers:",
      len(set(topics)) - (1 if -1 in topics else 0))

# Identify outlier reviews (topic -1)
outlier_indices = [
    i for i, topic in enumerate(topics) if topic == -1
]

print("Outlier reviews:", len(outlier_indices))
print(
    "Outlier percentage:",
    round(len(outlier_indices) / len(topics) * 100, 2), "%"
)

# Display the first 10 outlier reviews
for i in outlier_indices[:10]:
    print(f"\nReview {i}: {document[i]}")

,Topic,Count,Name,Representation,Representative_Docs
0,-1,4383,-1_overall_genuinely_staff_paid,"[overall, genuinely, staff, paid, stay, impres...",[we couldn t have asked for a better base whil...
1,0,11050,0_time_hotel_stay_fine,"[time, hotel, stay, fine, visit, expected, unr...",[this hotel was a fine if unremarkable choice ...
2,1,5602,1_overall_check_desk_paid,"[overall, check, desk, paid, arrival, expected...",[the housekeeping staff was fine nothing speci...
3,2,2905,2_breakfast_cafe_house cafe_house,"[breakfast, cafe, house cafe, house, buffet, o...",[i d describe the in house cafe as standard ho...
4,3,1433,3_gym_facilities_fitness_fitness facilities,"[gym, facilities, fitness, fitness facilities,...",[i d describe the fitness facilities as fine f...
5,4,1259,4_similar hotels_similar_hotels_rate,"[similar hotels, similar, hotels, rate, rate p...",[i d describe the cost of the stay as in line ...
6,5,1238,5_getting hotel_getting_hotel_transport,"[getting hotel, getting, hotel, transport, sur...",[i d describe getting around from the hotel as...
7,6,732,6_birmingham_stay_start_got,"[birmingham, stay, start, got, visit, ordinary...",[our trip to birmingham got off to a great sta...
8,7,719,7_business center_center_business_overall busi...,"[business center, center, business, overall bu...",[overall the business center was fine for our ...
9,8,708,8_pool_overall pool_paid_pool basic functional,"[pool, overall pool, paid, pool basic function...",[i d describe the pool as basic but functional...


Total reviews: 35000
Number of topics excluding outliers: 69
Outlier reviews: 4383
Outlier percentage: 12.52 %

Review 0: the overall price was fair for what we paid might consider it again might not

Review 3: we were disappointed that the neighborhood was disappointingly out of the way meanwhile sadly the room turned out to be grimy

Review 30: unfortunately the pricing was disappointing for the price several amenities cost extra on top of the rate not ideal for a anniversary trip

Review 33: overall the buffet was nothing to write home about about what we expected i d describe the location as fine nothing was within walking distance but transport was easy enough

Review 45: i was genuinely impressed that the dining options was a real highlight meanwhile i was genuinely impressed that the housekeeping staff was remarkably helpful

Review 63: the surrounding area was not the most convenient but manageable nothing more and nothing less on the other hand i d describe the overall dcor as

In [31]:
# Reassign outlier reviews using their embeddings
new_topics = topic_model.reduce_outliers(
    document,
    topics,
    strategy="embeddings",
    embeddings=embeddings
)

# Compare the original and updated outlier counts
print("Original outliers:", sum(t == -1 for t in topics))
print("Remaining outliers:", sum(t == -1 for t in new_topics))

print(
    "Updated outlier percentage:",
    round(sum(t == -1 for t in new_topics) / len(new_topics) * 100, 2),
    "%"
)

Original outliers: 4383
Remaining outliers: 0
Updated outlier percentage: 0.0 %


In [32]:
# Display the top words for each topic
for topic_id in topic_model.get_topic_info()["Topic"]:
    if topic_id == -1:
        continue

    print(f"\nTopic {topic_id}")
    print("Top words:", topic_model.get_topic(topic_id))

    # Display representative reviews, when available
    try:
        print(
            "Representative reviews:",
            topic_model.get_representative_docs(topic_id)[:3]
        )
    except (KeyError, ValueError):
        print("No representative reviews available.")


Topic 0
Top words: [('time', np.float64(0.025535951195838506)), ('hotel', np.float64(0.024490083080605143)), ('stay', np.float64(0.021831831065774097)), ('fine', np.float64(0.01679079179221947)), ('visit', np.float64(0.015891511436467557)), ('expected', np.float64(0.015085260927397856)), ('unremarkable', np.float64(0.014495332306721963)), ('london', np.float64(0.014400065477488665)), ('overall', np.float64(0.014276209778031252)), ('hotel fine', np.float64(0.01406310138384382))]
Representative reviews: ['this hotel was a fine if unremarkable choice for our time in birmingham also the neighborhood was acceptable for what we paid', 'this hotel was a fine if unremarkable choice for our time in london meanwhile the business center was standard nothing more and nothing less', 'this hotel was a fine if unremarkable choice for our time in liverpool on top of that the check in process was acceptable nothing more and nothing less']

Topic 1
Top words: [('overall', np.float64(0.01813020314821200